In [0]:
from pyspark.sql import SparkSession 
from pyspark.sql.functions import*
spark=SparkSession.builder.getOrCreate()
data = [
(101,"Arjun Reddy","Hyderabad","Cardiology",5000,1),
(102,"Sneha Kapoor","Delhi","Orthopedics",3000,2),
(103,"Rahul Sharma","Mumbai","Dermatology",1500,1),
(104,"Priya Nair","Bangalore","Cardiology",5000,2),
(105,"Vikram Singh","Chennai","Neurology",7000,1),
(106,"Ananya Das","Kolkata","Orthopedics",3000,3),
(107,"Karan Patel","Ahmedabad","Cardiology",5000,1),
(108,"Meera Iyer","Bangalore","Dermatology",1500,2)
]
columns = [
"visit_id",
"patient_name",
"city",
"department",
"consultation_fee",
"tests_count"
]
df=spark.createDataFrame(data,columns)
df_derived=df.withColumn("total_bill",col("consultation_fee")+(col("tests_count")*500)).withColumn("patient_category",when(col("total_bill")>=6000,"Premiuim").when(col("total_bill")>=4000,"Standard").otherwise("Basic"))
high_value_patients=df_derived.filter(col("total_bill")>5000)
dept_agg=df_derived.groupBy("department").agg(count("*").alias("patient_count"),avg("total_bill").alias("avg_bill"),sum("total_bill").alias("total_revenue"))
sorted_results=dept_agg.orderBy(col("total_revenue").desc())
display(sorted_results)

department,patient_count,avg_bill,total_revenue
Cardiology,3,5666.666666666667,17000
Orthopedics,2,4250.0,8500
Neurology,1,7500.0,7500
Dermatology,2,2250.0,4500


In [0]:
df_derived.createOrReplaceTempView("hospital_visits")


In [0]:
%sql
    
SELECT * FROM hospital_visits WHERE department='Cardiology';

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category
101,Arjun Reddy,Hyderabad,Cardiology,5000,1,5500,Standard
104,Priya Nair,Bangalore,Cardiology,5000,2,6000,Premiuim
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Standard


In [0]:
%sql
SELECT city,SUM(total_bill) AS total_city_revenue FROM hospital_visits GROUP BY city ORDER BY total_city_revenue DESC;

city,total_city_revenue
Bangalore,8500
Chennai,7500
Hyderabad,5500
Ahmedabad,5500
Kolkata,4500
Delhi,4000
Mumbai,2000


In [0]:
%sql
SELECT patient_name,total_bill FROM hospital_visits ORDER BY total_bill DESC  LIMIT 3;

patient_name,total_bill
Vikram Singh,7500
Priya Nair,6000
Arjun Reddy,5500


In [0]:
%sql
SELECT department,COUNT(*) AS patient_count FROM hospital_visits GROUP BY department;

department,patient_count
Cardiology,3
Orthopedics,2
Dermatology,2
Neurology,1


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_data_delta(visit_id INT,patient_name STRING,city STRING,department STRING,consultation_fee INT,tests_count INT,total_bill INT,patient_category STRING)
USING DELTA;

In [0]:
%sql
INSERT INTO hospital_data_delta VALUES
(101,'Arjun Reddy','Hyderabad','Cariology',5000,1,5500,'Standard'),
(102,'Sneha kappor','Delhi','Orthopedics',3000,2,4000,'Standard'),
(103,'Rahul Sharma','Mumbai','Dermatology',1500,1,200,'Basic'),
(104,'Priya Nair','Bangalore','Cardiology',5000,2,6000,'Premium'),
(105,'Vikra Singh','Chennai','Neurology',7000,1,7500,'Premiuim'),
(106,'Ananya Das','Kolkata','Orthopedics',3000,3,4500,'Standard'),
(107,'Karan Patel','Ahmedabad','Cardiology',5000,1,5500,'Standard'),
(108,'Meera Iyer','Bangalore','Dermatology',1500,2,2500,'Basic');

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
INSERT INTO hospital_data_delta VALUES(109,'Riya Sen','Pune','Neurology',6500,2,7500,'Permium');

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql
UPDATE hospital_data_delta SET consultation_fee=3500,total_bill=4500 WHERE visit_id=102;

num_affected_rows
1


In [0]:
%sql
DELETE FROM hospital_data_delta WHERE visit_id=103;

num_affected_rows
1


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hospital_updates AS SELECT * FROM VALUES
(104,'Priya Nair','Bangalore','Cardiology',5500,2,6500,'Premium'),
(110,'Amit Shah','Delhi','Orthopedics',4000,1,4500,'Standard') AS hospital_updates(visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category);

In [0]:
%sql
MERGE INTO hospital_data_delta AS target USING hospital_updates AS source ON target.visit_id=source.visit_id 
WHEN MATCHED THEN UPDATE SET target.patient_name=source.patient_name,target.city=source.city,target.department=source.department,target.consultation_fee=source.consultation_fee,target.tests_count=source.tests_count,target.total_bill=source.total_bill,target.patient_category=source.patient_category 
WHEN NOT MATCHED THEN INSERT (visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category) VALUES (source.visit_id,source.patient_name,source.city,source.department,source.consultation_fee,source.tests_count,source.total_bill,source.patient_category);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM hospital_data_delta

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category
102,Sneha kappor,Delhi,Orthopedics,3500,2,4500,Standard
101,Arjun Reddy,Hyderabad,Cariology,5000,1,5500,Standard
105,Vikra Singh,Chennai,Neurology,7000,1,7500,Premiuim
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Standard
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Standard
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Basic
109,Riya Sen,Pune,Neurology,6500,2,7500,Permium
104,Priya Nair,Bangalore,Cardiology,5500,2,6500,Premium
110,Amit Shah,Delhi,Orthopedics,4000,1,4500,Standard


In [0]:
%sql
DESCRIBE HISTORY hospital_data_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
17,2026-05-04T06:22:52.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3984921269914374),d3094e8d-baa3-477a-be92-3d06c46c14e7,0504-040905-tp0bpvoc-v2n,16,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7214, p25FileSize -> 2798, numDeletionVectorsRemoved -> 1, minFileSize -> 2798, numAddedFiles -> 1, maxFileSize -> 2798, p75FileSize -> 2798, p50FileSize -> 2798, numAddedBytes -> 2798)",null,Databricks-Runtime/18.1.x-photon-scala2.13
16,2026-05-04T06:22:50.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(visit_id#25675 = visit_id#25659)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> true, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(3984921269914374),d3094e8d-baa3-477a-be92-3d06c46c14e7,0504-040905-tp0bpvoc-v2n,15,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 2, numTargetBytesAdded -> 4422, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 1, executionTimeMs -> 2589, materializeSourceTimeMs -> 98, numTargetRowsInserted -> 1, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 977, numTargetRowsUpdated -> 1, numOutputRows -> 2, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 2, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1481)",null,Databricks-Runtime/18.1.x-photon-scala2.13
15,2026-05-04T06:22:47.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3984921269914374),29895ad5-0e90-48f5-a635-5a6c2e78f24e,0504-040905-tp0bpvoc-v2n,14,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 2843, p25FileSize -> 2792, numDeletionVectorsRemoved -> 1, minFileSize -> 2792, numAddedFiles -> 1, maxFileSize -> 2792, p75FileSize -> 2792, p50FileSize -> 2792, numAddedBytes -> 2792)",null,Databricks-Runtime/18.1.x-photon-scala2.13
14,2026-05-04T06:22:46.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,DELETE,"Map(predicate -> [""(visit_id#25283 = 103)""])",null,List(3984921269914374),29895ad5-0e90-48f5-a635-5a6c2e78f24e,0504-040905-tp0bpvoc-v2n,12,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1213, conflictDetectionTimeMs -> 432, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 862, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 351)",null,Databricks-Runtime/18.1.x-photon-scala2.13
13,2026-05-04T06:22:44.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(3984921269914374),59fa001d-1fcf-413d-b6de-fbe10600b17b,0504-040905-tp0bpvoc-v2n,12,SnapshotIsolation,false,"Map(numRemovedFiles -> 3, numRemovedBytes -> 7144, p25FileSize -> 2843, numDeletionVectorsRemoved -> 1, minFileSize -> 2843, numAddedFiles -> 1, maxFileSize -> 2843, p75FileSize -> 2843, p50FileSize -> 2843, numAddedBytes -> 2843)",null,Databricks-Runtime/18.1.x-photon-scala2.13
12,2026-05-04T06:22:42.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,UPDATE,"Map(predicate -> [""(visit_id#24703 = 102)""])",null,List(3984921269914374),59f

In [0]:
%sql
SELECT * FROM hospital_data_delta VERSION AS OF 0;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category


VACUUM removes obsolete data files from older table versions after retention period.This saves storage but reduce time-travel capability for deleted versions

In [0]:
%sql
VACUUM hospital_data_delta RETAIN 168 HOURS DRY RUN;

path


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hospital_parquet_data AS SELECT * FROM hospital_data_delta;

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_delta_converted USING DELTA AS SELECT * FROM hospital_parquet_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE DETAIL hospital_delta_converted;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,91f44808-f683-4ab1-b01f-ae430e257908,hexa_ws_7405618743358946.default.hospital_delta_converted,null,,2026-05-04T06:23:04.068Z,2026-05-04T06:23:05.000Z,List(),List(),1,2796,"Map(delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW hospital_parquet_data AS SELECT * FROM hospital_data_delta;

In [0]:
%sql
CREATE OR REPLACE TABLE hospital_delta_converted USING DELTA AS SELECT * FROM hospital_parquet_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE DETAIL hospital_delta_converted;
SELECT * FROM hospital_delta_converted;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category
102,Sneha kappor,Delhi,Orthopedics,3500,2,4500,Standard
101,Arjun Reddy,Hyderabad,Cariology,5000,1,5500,Standard
105,Vikra Singh,Chennai,Neurology,7000,1,7500,Premiuim
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Standard
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Standard
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Basic
109,Riya Sen,Pune,Neurology,6500,2,7500,Permium
104,Priya Nair,Bangalore,Cardiology,5500,2,6500,Premium
110,Amit Shah,Delhi,Orthopedics,4000,1,4500,Standard


In [0]:
%sql
CREATE OR REPLACE TABLE target_hospital_data USING DELTA AS SELECT * FROM hospital_data_delta;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS SELECT * FROM VALUES(105,'Vikram Singh','Chennai','Neurology',7500,2,8500,'Premiuim'),(111,'Neha Joshi','Pune','Dermatology',2000,1,2500,'Basic') AS daily_updates(visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category);

In [0]:
%sql
    
MERGE INTO target_hospital_data AS target USING daily_updates AS source ON target.visit_id=source.visit_id 
WHEN MATCHED THEN UPDATE SET target.patient_name=source.patient_name,target.city=source.city,target.department=source.department,target.consultation_fee=source.consultation_fee,target.tests_count=source.tests_count,target.total_bill=source.total_bill,target.patient_category=source.patient_category WHEN NOT MATCHED THEN INSERT(visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category) VALUES(source.visit_id,source.patient_name,source.city,source.department,source.consultation_fee,source.tests_count,source.total_bill,source.patient_category);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
SELECT * FROM target_hospital_data ORDER BY visit_id;

visit_id,patient_name,city,department,consultation_fee,tests_count,total_bill,patient_category
101,Arjun Reddy,Hyderabad,Cariology,5000,1,5500,Standard
102,Sneha kappor,Delhi,Orthopedics,3500,2,4500,Standard
104,Priya Nair,Bangalore,Cardiology,5500,2,6500,Premium
105,Vikram Singh,Chennai,Neurology,7500,2,8500,Premiuim
106,Ananya Das,Kolkata,Orthopedics,3000,3,4500,Standard
107,Karan Patel,Ahmedabad,Cardiology,5000,1,5500,Standard
108,Meera Iyer,Bangalore,Dermatology,1500,2,2500,Basic
109,Riya Sen,Pune,Neurology,6500,2,7500,Permium
110,Amit Shah,Delhi,Orthopedics,4000,1,4500,Standard
111,Neha Joshi,Pune,Dermatology,2000,1,2500,Basic


In [0]:
%sql
CREATE CATALOG hospitals_catalogs;

In [0]:
%sql
CREATE SCHEMA hospitals_catalogs.health_schema;

In [0]:
%sql
CREATE OR REPLACE TABLE hospitals_catalogs.health_schema.patient_data USING DELTA AS SELECT * FROM hospital_data_delta;

num_affected_rows,num_inserted_rows


In [0]:
%sql
INSERT INTO hospitals_catalogs.health_schema.patient_data SELECT * FROM hospital_data_delta;

num_affected_rows,num_inserted_rows
9,9


In [0]:
%sql
SHOW CATALOGS;

catalog
assessment
bronze
datas
hexa_ws_7405618743358946
hexacatalog_student
hospitals_catalog
hospitals_catalogs
hospitalss_catalogs
medallion
retailss_catalogss


In [0]:
%sql
SHOW SCHEMAS IN hospitals_catalogs;

databaseName
default
health_schema
information_schema


In [0]:
%sql
SHOW TABLES IN hospitals_catalogs.health_schema;

database,tableName,isTemporary
health_schema,patient_data,false
,daily_updates,true
,hospital_parquet_data,true
,hospital_updates,true
,hospital_visits,true


In [0]:
%sql
CREATE OR REPLACE TABLE hospitals_catalogs.health_schema.city_revenue AS SELECT city,SUM(total_bill) AS total_revenue,COUNT(*) AS patient_count FROM hospitals_catalog.health_schema.patient_data GROUP BY city;

num_affected_rows,num_inserted_rows


In [0]:
%sql
    
GRANT SELECT ON TABLE hospitals_catalogs.health_schema.patient_data
TO `account users`;

In [0]:
%sql
DESCRIBE HISTORY hospitals_catalog.health_schema.patient_data;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-05-04T05:33:15.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(3984921269914374),8ecbd35d-b8da-4ecc-8680-cf16459ebfaa,0504-040905-tp0bpvoc-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 9, numOutputBytes -> 2798)",null,Databricks-Runtime/18.1.x-photon-scala2.13
0,2026-05-04T05:32:11.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3984921269914374),8f522aca-1a9a-4c5e-aad6-eaddb58cdbf5,0504-040905-tp0bpvoc-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 9, numOutputBytes -> 2794)",null,Databricks-Runtime/18.1.x-photon-scala2.13


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_raw_data USING DELTA AS
SELECT * FROM VALUES
(101,'Arjun Reddy','Hyderabad','Cardiology',5000,1),
(102,'Sneha Kapoor','Delhi','Orthopedics',3000,2),
(103,'Rahul Sharma','Mumbai','Dermatology',1500,1),
(104,'Priya Nair','Bangalore','Cardiology',5000,2),
(105,'Vikram Singh','Chennai','Neurology',7000,1),
(106,'Ananya Das','Kolkata','Orthopedics',3000,3),
(107,'Karan Patel','Ahmedabad','Cardiology',5000,1),
(108,'Meera Iyer','Bangalore','Dermatology',1500,2)
AS hospital_raw_data(
    visit_id,
    patient_name,
    city,
    department,
    consultation_fee,
    tests_count
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_clean_data USING DELTA AS
SELECT visit_id,patient_name,city,department,consultation_fee,tests_count,consultation_fee+(tests_count*500) AS total_bill,CASE WHEN consultation_fee+(tests_count*500)>=6000 THEN 'Premium' WHEN consultation_fee+(tests_count* 500)>=4000 THEN 'Standard' ELSE 'Basic' END AS patient_category FROM hospital_raw_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TABLE hospital_analytics USING DELTA AS SELECT department,COUNT(*) AS patient_count,SUM(total_bill) AS total_revenue,AVG(total_bill) AS avg_bill FROM hospital_clean_data GROUP BY department;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW daily_updates AS SELECT* FROM VALUES(105,'Vikram Singh','Chennai','Neurology',7500,2),
(111,'Neha Joshi','Pune','Dermatology',2000,1) AS daily_updates(visit_id,patient_name,city,department,consultation_fee,tests_count);

In [0]:
%sql
MERGE INTO hospital_raw_data AS target
USING daily_updates AS source
ON target.visit_id = source.visit_id
WHEN MATCHED THEN UPDATE SET target.patient_name = source.patient_name, target.city = source.city,target.department = source.department,target.consultation_fee = source.consultation_fee,target.tests_count = source.tests_count
WHEN NOT MATCHED THEN INSERT (visit_id,patient_name,city,department,consultation_fee,tests_count)
VALUES (source.visit_id,source.patient_name,source.city,source.department,source.consultation_fee,source.tests_count);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


In [0]:
%sql
CREATE CATALOG hospitalsss_catalogs;

In [0]:
%sql
CREATE SCHEMA hospitalsss_catalogs.health_schema;

In [0]:
%sql
CREATE OR REPLACE TABLE hospitalsss_catalogs.health_schema.governed_hospital_data
USING DELTA AS SELECT * FROM hospital_clean_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW TABLES IN hospitalsss_catalogs.health_schema;

database,tableName,isTemporary
health_schema,governed_hospital_data,false
,daily_updates,true
,hospital_parquet_data,true
,hospital_updates,true
,hospital_visits,true


In [0]:
%sql
CREATE OR REPLACE TABLE hospitalss_catalogs.health_schema.city_revenue AS
SELECT city,SUM(total_bill) AS total_revenue
FROM hospitalsss_catalogs.health_schema.governed_hospital_data
GROUP BY city;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE HISTORY hospitalsss_catalogs.health_schema.governed_hospital_data;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-05-04T06:26:37.000Z,146756435897876,azuser5810_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3984921269914374),89c099b7-e538-424f-a571-bdce1371177f,0504-040905-tp0bpvoc-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 8, numOutputBytes -> 2742)",null,Databricks-Runtime/18.1.x-photon-scala2.13
